In [ ]:
import numpy as np
import pandas as pd

# ==============================================================================
# EXPERIMENT: Pandas Library Use Cases in Data Science
# OBJECTIVE: Demonstrate core DataFrame operations including ingestion, inspection,
#            selection, cleaning, transformation, aggregation, and merging.
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. DATA INGESTION & INSPECTION
# ------------------------------------------------------------------------------
print("=== 1. DATA INGESTION & INSPECTION ===")

# Load Pima Indians Diabetes Dataset directly from URL
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]

df = pd.read_csv(url, header=None, names=columns)

print("First 5 rows:")
print(df.head())

print("\nDataset Metadata & Data Types:")
print(df.info())

print("\nSummary Statistics:")
print(df.describe().T)

# ------------------------------------------------------------------------------
# 2. DATA SELECTION, FILTERING & QUERYING
# ------------------------------------------------------------------------------
print("\n=== 2. SELECTION & FILTERING ===")

# Column Selection
subset = df[["Age", "Glucose", "Outcome"]]

# Label-based (loc) vs Position-based (iloc) indexing
print("\nRow 0-4 using iloc:")
print(df.iloc[0:5, 0:3])

# Boolean Indexing / Conditional Filtering: High Risk Group
high_risk = df[(df["Glucose"] > 140) & (df["Age"] > 35) & (df["Outcome"] == 1)]
print(
    f"\nNumber of high-risk patients (Glucose > 140, Age > 35, Outcome = 1): {len(high_risk)}"
)

# ------------------------------------------------------------------------------
# 3. DATA CLEANING & TRANSFORMATION
# ------------------------------------------------------------------------------
print("\n=== 3. DATA CLEANING & TRANSFORMATION ===")

# In this dataset, zero values in Glucose, BloodPressure, BMI represent missing data
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_cols] = df[zero_cols].replace(0, np.nan)

print("\nMissing values per column after replacing 0s with NaN:")
print(df.isnull().sum())

# Handle Missing Values: Impute missing values with column median
df.fillna(df.median(), inplace=True)

# Feature Engineering / Column Creation: Create Age Group Categories
bins = [20, 30, 45, 60, 100]
labels = ["Young Adult", "Middle-Aged", "Senior", "Elderly"]
df["Age_Group"] = pd.cut(df["Age"], bins=bins, labels=labels)

print("\nProcessed Dataset with Age_Group feature:")
print(df[["Age", "Age_Group", "BMI", "Glucose"]].head())

# ------------------------------------------------------------------------------
# 4. GROUPING & AGGREGATION
# ------------------------------------------------------------------------------
print("\n=== 4. GROUPING & AGGREGATION ===")

# GroupBy Outcome to see mean feature values
print("\nMean metrics by Diabetes Outcome (0 = Negative, 1 = Positive):")
print(df.groupby("Outcome")[["Glucose", "BMI", "Age", "Insulin"]].mean())

# Complex Aggregation by Age Group & Outcome
grouped_summary = df.groupby(["Age_Group", "Outcome"], observed=False).agg(
    Avg_Glucose=("Glucose", "mean"),
    Avg_BMI=("BMI", "mean"),
    Patient_Count=("Outcome", "count"),
)
print("\nMulti-level Aggregation Summary:")
print(grouped_summary)

# Pivot Table: Average Glucose by Age Group and Outcome
pivot_tbl = pd.pivot_table(
    df,
    values="Glucose",
    index="Age_Group",
    columns="Outcome",
    aggfunc="mean",
    observed=False,
)
print("\nPivot Table (Mean Glucose by Age Group vs Outcome):")
print(pivot_tbl)

# ------------------------------------------------------------------------------
# 5. MERGING & COMBINING DATAFRAMES
# ------------------------------------------------------------------------------
print("\n=== 5. MERGING & COMBINING DATAFRAMES ===")

# Create a mock lookup table for Risk Categories based on Outcome
risk_lookup = pd.DataFrame(
    {
        "Outcome": [0, 1],
        "Risk_Category": ["Low Risk", "High Risk"],
        "Recommended_Action": ["Annual Checkup", "Follow-up Specialist"],
    }
)

# Merge main dataframe with lookup table
df_merged = pd.merge(df, risk_lookup, on="Outcome", how="left")
print("\nSample Rows from Merged Dataset:")
print(
    df_merged[
        [
            "Glucose",
            "BMI",
            "Age",
            "Outcome",
            "Risk_Category",
            "Recommended_Action",
        ]
    ].head()
)